# Lab 3 — European Rain Forecast: Score Your Model Locally

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/data_science_practice/blob/main/website/public/modules/ms2a-machine-learning-practice/challenges/mlp-s3-rain-model.ipynb)

A forecast submitted to challenge 177 is scored 48 hours later. This
notebook replays the challenge on the past instead: every forecast
issued from 2025-01-01 to 2026-03-01, as the exact request the
challenge sends, scored the way the challenge scores it.

It scores two agents, the city × month climatology of the agent
notebook and a random forest, then leaves the model to you.

---

## 1. Setup and download

The notebook needs one secret, `MLARENA_API_KEY`: ML-Arena, Profile →
API Keys (it starts with `mlk_user_`). Never paste its value into a
cell: a notebook is shared with its code and its outputs.

In Colab, add it to the *Secrets* panel (the key icon on the left) and
allow this notebook to access it. Outside Colab, set it in your
environment before starting Jupyter; the cell raises if it is missing.
Colab already has pandas, numpy and scikit-learn; the only install
is the ML-Arena client.

In [ ]:
!pip install -q mlarena-sdk

In [ ]:
import os

MLARENA_API_KEY = "<your_api_key>"

import mlarena
import numpy as np
import pandas as pd
import requests
from sklearn.ensemble import RandomForestRegressor

client = mlarena.connect(api_key=MLARENA_API_KEY)
CHALLENGE_ID = 177

In [ ]:
FILE = "weather_europe_2020_2026.csv.gz"

if not os.path.exists(FILE):
    listing = client.datasets(CHALLENGE_ID)
    [meta] = [f for ds in listing["datasets"] for f in ds["files"]
              if f["label"] == FILE]
    resp = requests.get(meta["download_url"], timeout=300)
    resp.raise_for_status()
    with open(FILE, "wb") as fh:
        fh.write(resp.content)
print(FILE, f"{os.path.getsize(FILE) / 1e6:.1f} MB")

---

## 2. The data, as the challenge sees it

The challenge sends the cities in its own fixed order, not the
file's alphabetical one. `X` is the whole file as an array (hours,
cities, features) in that order, so a request is a slice of it.

In [ ]:
PANEL = [
    ("Amsterdam", "NL"), ("Athens", "GR"), ("Belgrade", "RS"),
    ("Berlin", "DE"), ("Brussels", "BE"), ("Bucharest", "RO"),
    ("Budapest", "HU"), ("Chisinau", "MD"), ("Copenhagen", "DK"),
    ("Dublin", "IE"), ("Helsinki", "FI"), ("Kyiv", "UA"),
    ("London", "GB"), ("Madrid", "ES"), ("Minsk", "BY"),
    ("Moscow", "RU"), ("Oslo", "NO"), ("Paris", "FR"),
    ("Prague", "CZ"), ("Riga", "LV"), ("Rome", "IT"),
    ("Sarajevo", "BA"), ("Sofia", "BG"), ("Stockholm", "SE"),
    ("Vienna", "AT"), ("Warsaw", "PL"), ("Zagreb", "HR"),
    ("Istanbul", "TR"), ("Saint Petersburg", "RU"), ("Hamburg", "DE"),
    ("Munich", "DE"), ("Frankfurt am Main", "DE"), ("Milan", "IT"),
    ("Naples", "IT"), ("Palermo", "IT"), ("Barcelona", "ES"),
    ("Valencia", "ES"), ("Sevilla", "ES"), ("Marseille", "FR"),
    ("Birmingham", "GB"), ("Glasgow", "GB"), ("Kraków", "PL"),
    ("Göteborg", "SE"), ("Odesa", "UA"), ("Kharkiv", "UA"),
]
FEATURES = ["temperature", "rain", "wind_speed", "wind_direction",
            "humidity", "clouds", "visibility", "snow"]

df = pd.read_csv(FILE, dtype={"city_name": "category",
                              "country_code": "category"})
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True,
                                 format="ISO8601")
key = df["city_name"].astype(str) + "|" + df["country_code"].astype(str)
position = {f"{n}|{c}": j for j, (n, c) in enumerate(PANEL)}
df["pos"] = key.map(position)
assert df["pos"].notna().all() and df["pos"].nunique() == len(PANEL)
df = df.sort_values(["timestamp", "pos"])

hours = pd.DatetimeIndex(df["timestamp"].unique())
assert (hours[1:] - hours[:-1] == pd.Timedelta(hours=1)).all()
X = df[FEATURES].to_numpy(float).reshape(len(hours), len(PANEL),
                                         len(FEATURES))
coords = df.groupby("pos")[["latitude", "longitude"]].first()
rain = X[:, :, FEATURES.index("rain")]          # (hours, cities)
month = hours.month.to_numpy()
del df, key
print(X.shape, hours[0], "->", hours[-1])

Models learn from **2020–2024**. The replay covers the forecasts
issued from **2025-01-01**, every 6 hours at the UTC hours the
challenge uses, up to the last one whose +48 h is in the file.
`issue_times(start, end)` keeps the issue hours whose +48 h falls
before `end`, so no training target reaches into the replay.

In [ ]:
HORIZONS = [6, 48]
ISSUE_HOURS = (5, 11, 17, 23)          # UTC, every 6 hours
SPLIT = hours.searchsorted(pd.Timestamp("2025-01-01", tz="UTC"))


def issue_times(start, end):
    """Issue hours in [start, end) with 48 h of history and their +48 h."""
    i = np.arange(max(start, 47), end - max(HORIZONS))
    return i[np.isin(hours.hour[i], ISSUE_HOURS)]


TRAIN = issue_times(0, SPLIT)
TEST = issue_times(SPLIT, len(hours))
print(len(TRAIN), "training issue times |", len(TEST), "replayed,",
      hours[TEST[0]], "->", hours[TEST[-1]])

---

## 3. The request and the local score

`make_request(i)` builds what the challenge sends when hour `i` is
the issue time (the keys are in the agent notebook, section 1): the
48 hours ending at `i`, the 45 cities in panel order, the 8 features.
The feed never reports `visibility`, and the challenge sends it as 0.
`check` validates a response the way the challenge does, and `crps`
is the score of the analysis notebook, section 9.

In [ ]:
import json


def iso(ts):
    return ts.strftime("%Y-%m-%dT%H:%M:%SZ")


def window(i):
    """The history sent at issue hour i: (cities, 48 h, features)."""
    return np.nan_to_num(X[i - 47:i + 1].transpose(1, 0, 2), nan=0.0)


def make_request(i):
    return {
        "issue_time": iso(hours[i]),
        "timestamps": [iso(t) for t in hours[i - 47:i + 1]],
        "observed": [True] * 48,
        "cities": [{"name": n, "country": c,
                    "lat": float(coords.loc[j, "latitude"]),
                    "lon": float(coords.loc[j, "longitude"])}
                   for j, (n, c) in enumerate(PANEL)],
        "feature_names": FEATURES,
        "history": window(i).tolist(),
        "horizons": HORIZONS,
        "max_members": 100,
    }


def check(response, request):
    """The response as a (cities, horizons, M) array, or raise."""
    json.dumps(response, allow_nan=False)      # JSON, no NaN/inf
    ens = np.asarray(response["rain"], dtype=float)
    shape = (len(request["cities"]), len(request["horizons"]))
    assert ens.ndim == 3 and ens.shape[:2] == shape, ens.shape
    assert 1 <= ens.shape[2] <= request["max_members"], ens.shape
    assert np.isfinite(ens).all()
    return ens


def crps(members, y):
    """CRPS of ensembles (..., M) against outcomes (...), in mm."""
    x = np.sort(np.maximum(members, 0.0), axis=-1)  # clipped at 0
    M = x.shape[-1]
    k = np.arange(1, M + 1)
    spread = (x * (2 * k - M - 1)).sum(axis=-1) / M ** 2
    return np.abs(x - y[..., None]).mean(axis=-1) - spread

`evaluate(agent, issues)` replays the challenge: for each issue time,
build the request, call `agent.predict`, and score both horizons
against the rain that fell. One row per run; the score is the mean
of the two CRPS, in mm, lower is better.

Consecutive runs share the weather, and the weather moves every
agent's CRPS together. `compare(a, b)` therefore compares two agents
run by run, and takes its standard error from weekly blocks of 28
runs.

In [ ]:
def evaluate(agent, issues=TEST):
    """One row per run: CRPS per horizon (mean over cities), score."""
    rows = []
    for i in issues:
        request = make_request(i)
        ens = check(agent.predict(request), request)
        rows.append({f"CRPS {h}h": crps(ens[:, k], rain[i + h]).mean()
                     for k, h in enumerate(HORIZONS)})
    runs = pd.DataFrame(rows, index=hours[issues])
    runs["score"] = runs.mean(axis=1)
    return runs


def compare(a, b, week=28):
    """Mean of a - b over the same runs, and its weekly-block SE."""
    d = (a["score"] - b["score"]).to_numpy()
    blocks = d[:len(d) // week * week].reshape(-1, week).mean(axis=1)
    return d.mean(), blocks.std(ddof=1) / np.sqrt(len(blocks))

---

## 4. The baseline: city × month climatology

The agent notebook's model, refitted on 2020–2024 only: a table
that has seen the replayed months would score itself on its own
data.

In [ ]:
M = 100
LEVELS = (np.arange(M) + 0.5) / M


class Climatology:
    """For each city and valid month, M quantiles of the past rain."""

    def __init__(self, end):
        r, m = rain[:end], month[:end]
        self.table = np.stack([np.quantile(r[m == k], LEVELS, axis=0).T
                               for k in range(1, 13)])  # (12, cities, M)

    def predict(self, request):
        issue = pd.Timestamp(request["issue_time"])
        months = [(issue + pd.Timedelta(hours=h)).month
                  for h in request["horizons"]]
        members = self.table[np.array(months) - 1]    # (2, cities, M)
        return {"rain": members.transpose(1, 0, 2).tolist()}


runs_clim = evaluate(Climatology(SPLIT))
print(runs_clim.mean().round(4).to_string())

---

## 5. A random forest

Five features per city, read from the request: the rain now and
over the last 6 hours, the humidity and the cloud cover now, and the
month. The same `features` function builds the training rows from
`window(i)` and reads the live request, so the model sees at
prediction exactly what it saw in training.

In [ ]:
RAIN, HUMIDITY, CLOUDS = (FEATURES.index(f)
                          for f in ("rain", "humidity", "clouds"))


def features(history, issue):
    """(cities, 5) features from a history (cities, 48 h, features)."""
    r = history[:, :, RAIN]
    return np.column_stack([
        r[:, -1],                          # rain now
        r[:, -6:].sum(axis=1),             # rain over the last 6 h
        history[:, -1, HUMIDITY],
        history[:, -1, CLOUDS],
        np.full(len(history), issue.month),
    ])


F = np.concatenate([features(window(i), hours[i]) for i in TRAIN])
print(F.shape, "training rows: issue times x cities")

One forest per horizon. A forest is an ensemble already: each of its
30 trees gives one member.

In [ ]:
forests = {
    h: RandomForestRegressor(n_estimators=30, min_samples_leaf=100,
                             n_jobs=-1, random_state=0)
    .fit(F, np.concatenate([rain[i + h] for i in TRAIN]))
    for h in HORIZONS
}


class Forest:
    def __init__(self, forests):
        self.forests = forests

    def predict(self, request):
        F = features(np.asarray(request["history"]),
                     pd.Timestamp(request["issue_time"]))
        members = [np.stack([tree.predict(F)
                             for tree in self.forests[h].estimators_],
                            axis=-1)
                   for h in request["horizons"]]    # 2 x (cities, 30)
        return {"rain": np.stack(members, axis=1).tolist()}


runs_rf = evaluate(Forest(forests))
print(runs_rf.mean().round(4).to_string())
print("forest - climatology: %+.4f ± %.4f" % compare(runs_rf, runs_clim))

**Finding.** The forest scores 0.1076 (0.1020 at +6 h, 0.1132 at
+48 h): 0.0317 ± 0.0006 worse than the climatology, and worse than
forecasting 0 mm everywhere (0.0804, the analysis notebook). It reads
the weather; the climatology reads nothing.

**Question.** Why? Each tree predicts the mean of its leaf. Compare
the forest's members with the climatology's for one city and one run
— how many are 0, how far apart they are — and say which term of the
CRPS the forest loses on.

*Your answer:*

---

## 6. Your turn

Build features, fit a model, and score it here before you submit it.

- **Features.** Anything computed from the request. The analysis
  notebook lists the candidates: rain in the city upwind (section 7),
  the solar hour in summer (section 5), a 6-hour change with the time
  of day beside it (section 8). Write them in `features`: `window(i)`
  gives it in training exactly what the request gives it live.
- **Model.** Anything whose `predict(request)` returns (cities, 2, M)
  members, 1 ≤ M ≤ 100. The members are a distribution: a spike at
  0 and a long tail. A hurdle model (a classifier for P(wet), a
  distribution of amounts given wet, its quantiles as members) or one
  quantile regressor per level both produce one.
- **Evaluate.** `evaluate(agent, issues)`, then `compare` with the
  climatology on the same runs. A gap is real when it is several
  times its ±.

The bars, over the same replay: the climatology, 0.0759; the
challenge's starter agent (the template on the challenge page),
0.0746; a +6 h climatology split by rain in the last 3 hours, with
this climatology at +48 h, 0.0742.

Every look at the replay is a look at the test period. Tune on
2024, fitted on 2020–2023 (the cell below), and replay 2025–2026 once
per model you would submit, refitted on 2020–2024. To submit, write
`agent.py` as in the agent notebook, save the models next to it
(`joblib.dump`), and submit with `runtime_id=182`, the runtime with
scikit-learn, LightGBM and XGBoost.

In [ ]:
VALID = hours.searchsorted(pd.Timestamp("2024-01-01", tz="UTC"))
FIT, VAL = issue_times(0, VALID), issue_times(VALID, SPLIT)
clim_val = evaluate(Climatology(VALID), VAL)
print(clim_val.mean().round(4).to_string())

# Your model: fit on FIT only, then
# mine = ...                          # anything with .predict(request)
# runs_mine = evaluate(mine, VAL)
# print("mine - climatology: %+.4f ± %.4f" % compare(runs_mine, clim_val))